# SOPR Capitulation Signal - Fixed Exit Strategy

**Problem with v1:** Exited when SOPR recovered, missing the rally.

**Fix:** Use fixed holding period after capitulation entry.

## Exit Strategies to Test
1. Fixed hold period (7, 14, 30, 60 days)
2. Trailing stop
3. Target profit exit

In [ ]:
import pandas as pd
import numpy as np
import vectorbt as vbt
import statsmodels.api as sm
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

vbt.settings.plotting['use_widgets'] = False
vbt.settings.array_wrapper['freq'] = 'D'
vbt.settings.portfolio['init_cash'] = 100_000

print(f"VectorBT version: {vbt.__version__}")

In [ ]:
# Load data
DATA_DIR = Path("../data/daily")

sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")

df = sopr.join(sopr_sth, how='inner').join(price, how='inner').sort_index()

# Filter to 2019+
df = df[df.index >= '2018-12-15']

close = df['price']

print(f"Data: {len(df)} rows, {df.index.min().date()} to {df.index.max().date()}")

In [ ]:
# Create capitulation signal
# Entry: FIRST day when both SOPR < 1 (not every day)

both_below_1 = (df['sopr'] < 1) & (df['sopr_sth'] < 1)

# Entry = transition INTO capitulation (was not in signal, now is)
entries = both_below_1 & ~both_below_1.shift(1).fillna(False)

print(f"Capitulation entry signals: {entries.sum()}")
print(f"\nEntry dates:")
for date in entries[entries].index[:20]:
    print(f"  {date.date()} - Price: ${close.loc[date]:,.0f}")

---
## Strategy 1: Fixed Holding Period

Enter on capitulation signal, hold for N days, then exit.

In [ ]:
# Test different holding periods
hold_periods = [7, 14, 30, 60, 90]

results = []

for hold_days in hold_periods:
    # Create exit signal: N days after entry
    exits = entries.shift(hold_days).fillna(False)
    
    # Run backtest
    pf = vbt.Portfolio.from_signals(
        close=close,
        entries=entries,
        exits=exits,
        init_cash=100_000,
        fees=0.001,
        freq='D'
    )
    
    pf_hold = vbt.Portfolio.from_holding(close, init_cash=100_000, freq='D')
    
    results.append({
        'hold_days': hold_days,
        'total_return': pf.total_return(),
        'sharpe': pf.sharpe_ratio(),
        'max_dd': pf.max_drawdown(),
        'win_rate': pf.trades.win_rate() if pf.trades.count() > 0 else np.nan,
        'n_trades': pf.trades.count(),
        'excess_return': pf.total_return() - pf_hold.total_return()
    })

results_df = pd.DataFrame(results)
print("\nFIXED HOLDING PERIOD RESULTS")
print("="*70)
print(results_df.to_string(index=False))

In [ ]:
# Visualize
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=['Total Return', 'Sharpe Ratio', 'Win Rate', 'Excess vs B&H'])

fig.add_trace(go.Bar(x=[str(x) for x in results_df['hold_days']], 
                     y=results_df['total_return']*100, name='Return'), row=1, col=1)
fig.add_trace(go.Bar(x=[str(x) for x in results_df['hold_days']], 
                     y=results_df['sharpe'], name='Sharpe'), row=1, col=2)
fig.add_trace(go.Bar(x=[str(x) for x in results_df['hold_days']], 
                     y=results_df['win_rate']*100, name='Win %'), row=2, col=1)

colors = ['green' if x > 0 else 'red' for x in results_df['excess_return']]
fig.add_trace(go.Bar(x=[str(x) for x in results_df['hold_days']], 
                     y=results_df['excess_return']*100, marker_color=colors, name='Excess'), row=2, col=2)
fig.add_hline(y=0, line_dash='dash', row=2, col=2)

fig.update_layout(height=600, showlegend=False, title_text='Performance by Hold Period (days)')
fig.update_xaxes(title_text='Hold Days')
fig.show()

In [ ]:
# Best holding period
best_idx = results_df['sharpe'].idxmax()
best = results_df.loc[best_idx]

print(f"\nBEST HOLDING PERIOD: {best['hold_days']:.0f} days")
print(f"  Sharpe: {best['sharpe']:.2f}")
print(f"  Return: {best['total_return']*100:.1f}%")
print(f"  Win Rate: {best['win_rate']*100:.1f}%")
print(f"  Excess vs B&H: {best['excess_return']*100:+.1f}%")

---
## Visualize Best Strategy

In [ ]:
# Run best strategy
BEST_HOLD = int(best['hold_days'])

exits_best = entries.shift(BEST_HOLD).fillna(False)

pf_best = vbt.Portfolio.from_signals(
    close=close,
    entries=entries,
    exits=exits_best,
    init_cash=100_000,
    fees=0.001,
    freq='D'
)

pf_hold = vbt.Portfolio.from_holding(close, init_cash=100_000, freq='D')

print(f"Strategy: Buy on double capitulation, hold {BEST_HOLD} days")
print("\n" + "="*50)
print(pf_best.stats())

In [ ]:
# Plot entry/exit points clearly
fig = go.Figure()

# Price
fig.add_trace(go.Scatter(x=close.index, y=close, name='BTC Price', 
                         line=dict(color='blue', width=1)))

# Get actual trade entry/exit points from portfolio
if pf_best.trades.count() > 0:
    trades = pf_best.trades.records_readable
    
    # Entry points
    entry_dates = pd.to_datetime(trades['Entry Timestamp'])
    entry_prices = [close.loc[d] if d in close.index else close.asof(d) for d in entry_dates]
    
    fig.add_trace(go.Scatter(
        x=entry_dates, y=entry_prices,
        mode='markers',
        marker=dict(symbol='triangle-up', size=15, color='green', line=dict(width=2, color='darkgreen')),
        name=f'Entry (Capitulation)'
    ))
    
    # Exit points
    exit_dates = pd.to_datetime(trades['Exit Timestamp'])
    exit_prices = [close.loc[d] if d in close.index else close.asof(d) for d in exit_dates]
    
    fig.add_trace(go.Scatter(
        x=exit_dates, y=exit_prices,
        mode='markers',
        marker=dict(symbol='triangle-down', size=15, color='red', line=dict(width=2, color='darkred')),
        name=f'Exit ({BEST_HOLD} days later)'
    ))
    
    # Draw lines connecting entry to exit
    for i in range(len(trades)):
        color = 'green' if trades.iloc[i]['PnL'] > 0 else 'red'
        fig.add_trace(go.Scatter(
            x=[entry_dates.iloc[i], exit_dates.iloc[i]],
            y=[entry_prices[i], exit_prices[i]],
            mode='lines',
            line=dict(color=color, width=2, dash='dot'),
            showlegend=False,
            opacity=0.5
        ))

fig.update_layout(
    title=f'Double Capitulation Strategy: Entry + {BEST_HOLD}-Day Hold',
    yaxis_title='Price (USD)',
    yaxis_type='log',
    height=600,
    hovermode='x unified'
)
fig.show()

In [ ]:
# Trade details
if pf_best.trades.count() > 0:
    print("\nTRADE DETAILS")
    print("="*80)
    trades = pf_best.trades.records_readable
    trades['Return %'] = (trades['Return'] * 100).round(1)
    trades['PnL'] = trades['PnL'].round(0)
    print(trades[['Entry Timestamp', 'Exit Timestamp', 'Return %', 'PnL', 'Duration']].to_string())
    
    print(f"\n\nSUMMARY")
    print(f"  Total trades: {len(trades)}")
    print(f"  Winners: {(trades['Return'] > 0).sum()} ({(trades['Return'] > 0).mean()*100:.0f}%)")
    print(f"  Losers: {(trades['Return'] <= 0).sum()} ({(trades['Return'] <= 0).mean()*100:.0f}%)")
    print(f"  Avg win: {trades[trades['Return'] > 0]['Return'].mean()*100:.1f}%")
    print(f"  Avg loss: {trades[trades['Return'] <= 0]['Return'].mean()*100:.1f}%")

In [ ]:
# Equity curves
fig = go.Figure()

fig.add_trace(go.Scatter(x=pf_best.value().index, y=pf_best.value(), 
                         name=f'Capitulation + {BEST_HOLD}d Hold'))
fig.add_trace(go.Scatter(x=pf_hold.value().index, y=pf_hold.value(), 
                         name='Buy & Hold'))

fig.update_layout(title='Equity Curves', yaxis_title='Portfolio Value ($)', height=500)
fig.show()

---
## Walk-Forward Validation

In [ ]:
# Walk-forward with the best hold period
TRAIN_DAYS = 365
TEST_DAYS = 90
STEP_DAYS = 90

wf_results = []

total_days = len(close)
n_folds = (total_days - TRAIN_DAYS) // STEP_DAYS

for fold in range(n_folds):
    test_start = TRAIN_DAYS + fold * STEP_DAYS
    test_end = min(test_start + TEST_DAYS, total_days)
    
    if test_end <= test_start:
        break
    
    test_close = close.iloc[test_start:test_end]
    test_entries = entries.iloc[test_start:test_end]
    test_exits = test_entries.shift(BEST_HOLD).fillna(False)
    
    try:
        pf_test = vbt.Portfolio.from_signals(
            close=test_close, entries=test_entries, exits=test_exits,
            init_cash=100_000, fees=0.001, freq='D'
        )
        pf_hold_test = vbt.Portfolio.from_holding(test_close, init_cash=100_000, freq='D')
        
        test_return = pf_test.total_return()
        hold_return = pf_hold_test.total_return()
        n_trades = pf_test.trades.count()
    except:
        continue
    
    wf_results.append({
        'fold': fold,
        'period': close.index[test_start].strftime('%Y-%m'),
        'n_trades': n_trades,
        'strat_return': test_return,
        'hold_return': hold_return,
        'excess': test_return - hold_return,
        'beat_hold': test_return > hold_return
    })
    
    status = '✓' if test_return > hold_return else '✗'
    print(f"Fold {fold}: {close.index[test_start].strftime('%Y-%m')} | "
          f"{n_trades} trades | "
          f"Strat: {test_return*100:+6.1f}% | "
          f"B&H: {hold_return*100:+6.1f}% | {status}")

wf_df = pd.DataFrame(wf_results)

In [ ]:
# Walk-forward summary
if len(wf_df) > 0:
    print("\n" + "="*60)
    print("WALK-FORWARD RESULTS")
    print("="*60)
    
    wf_with_trades = wf_df[wf_df['n_trades'] > 0]
    
    print(f"\n{'Metric':<35} {'Value':>15}")
    print("-"*55)
    print(f"{'Total Folds':<35} {len(wf_df):>15}")
    print(f"{'Folds with Trades':<35} {len(wf_with_trades):>15}")
    print(f"{'Beat Buy&Hold (all)':<35} {wf_df['beat_hold'].mean()*100:>14.1f}%")
    
    if len(wf_with_trades) > 0:
        print(f"{'Beat Buy&Hold (w/ trades)':<35} {wf_with_trades['beat_hold'].mean()*100:>14.1f}%")
        print(f"{'Avg Excess Return':<35} {wf_with_trades['excess'].mean()*100:>+14.1f}%")

---
## Final Summary

In [ ]:
print("\n" + "="*70)
print("DOUBLE CAPITULATION SIGNAL - FINAL RESULTS")
print("="*70)

print(f"\n📊 SIGNAL")
print(f"   Entry: When SOPR < 1 AND STH SOPR < 1")
print(f"   Exit: {BEST_HOLD} days after entry (fixed hold)")

print(f"\n📈 IN-SAMPLE PERFORMANCE")
print(f"   Total Return: {pf_best.total_return()*100:.1f}%")
print(f"   Sharpe: {pf_best.sharpe_ratio():.2f}")
print(f"   Win Rate: {pf_best.trades.win_rate()*100:.0f}%")
print(f"   vs Buy&Hold: {(pf_best.total_return() - pf_hold.total_return())*100:+.1f}%")

if len(wf_df) > 0:
    print(f"\n🔍 WALK-FORWARD VALIDATION")
    print(f"   Beat Buy&Hold: {wf_df['beat_hold'].mean()*100:.0f}%")
    
    if wf_df['beat_hold'].mean() > 0.5:
        verdict = "✓ SIGNAL WORKS"
    else:
        verdict = "✗ SIGNAL DOESN'T BEAT BUY & HOLD"
    
    print(f"\n🎯 VERDICT: {verdict}")

print("\n" + "="*70)

In [ ]:
# Save results
import json

results_summary = {
    'signal': 'double_capitulation_fixed_hold',
    'entry': 'SOPR < 1 AND STH_SOPR < 1',
    'exit': f'{BEST_HOLD}_day_hold',
    'hold_days': BEST_HOLD,
    'in_sample_return': float(pf_best.total_return()),
    'in_sample_sharpe': float(pf_best.sharpe_ratio()),
    'win_rate': float(pf_best.trades.win_rate()) if pf_best.trades.count() > 0 else None,
    'n_trades': int(pf_best.trades.count()),
    'beat_hold_pct': float(wf_df['beat_hold'].mean()) if len(wf_df) > 0 else None,
    'n_folds': len(wf_df)
}

with open('../data/sopr_fixed_hold_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print("Saved to ../data/sopr_fixed_hold_results.json")
print(json.dumps(results_summary, indent=2))